## Data Extraction

In [ ]:
import pandas as pd

df = pd.read_csv('european_housing_prices_clean.csv')
#print(df.head())

country_list = df['country'].unique()
#print(country_list)

#Remove countries with Euro area or European Union in the name
countries_to_remove = [country for country in country_list if 'Euro area' in country or 'European Union' in country or 'Türkiye' in country]
#print(countries_to_remove)
df = df[~df['country'].isin(countries_to_remove)]
country_list = df['country'].unique()
#print(country_list)

#Change Czechia to Czech Republic
df['country'] = df['country'].replace('Czechia', 'Czech Republic')
country_list = df['country'].unique()
print(country_list)

df['dt'] = df['year'].astype(str) + '-0' + df['quarter_num'].astype(str) + '-01'
df['dt'] = pd.to_datetime(df['dt'], format='%Y-%m-%d')

['Austria' 'Belgium' 'Bulgaria' 'Croatia' 'Cyprus' 'Czech Republic'
 'Denmark' 'Estonia' 'Finland' 'France' 'Germany' 'Hungary' 'Iceland'
 'Ireland' 'Italy' 'Latvia' 'Lithuania' 'Luxembourg' 'Malta' 'Netherlands'
 'Norway' 'Poland' 'Portugal' 'Romania' 'Slovakia' 'Slovenia' 'Spain'
 'Sweden' 'Switzerland']


In [2]:
import plotly.express as px

# 1. Ensure the data is sorted chronologically for the slider
df = df.sort_values(by='quarter') # Replace with your actual column name

fig = px.choropleth(
    df, 
    locations='country', 
    locationmode='country names', 
    color='price_index',
    color_continuous_scale=['green', 'yellow', 'red'],
    hover_data={'price_index': ':.2f', 'quarter': True},
    animation_frame='quarter', # The slider will use your "2022-Q1" column
    range_color=[df['price_index'].min(), 300], # Keep scale consistent
    title='European Housing Price Index Over Time'
)

fig.update_layout(
    autosize=True,
    width=800,
    height=700, # Increased height slightly to accommodate the slider/play button
    margin={"r":0, "t":50, "l":0, "b":0},
    geo=dict(
        scope='europe',
        projection_type='mercator',
        lataxis_range=[34, 70], 
        lonaxis_range=[-20, 40],
        landcolor="#f3f5f2",
        showland=True,
        showocean=True,
        oceancolor="#e8f4f8",
        showcountries=True,
        showlakes=True,
        lakecolor="#e8f4f8",
        showframe=False,
        resolution=50,
    ),
    
)

fig.show()
fig.write_html('european_housing_prices_choropleth.html')

C:\Users\praga\AppData\Local\Temp\ipykernel_5564\657663353.py:6: DeprecationWarning: The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.
  fig = px.choropleth(


In [4]:
import folium
import requests
import plotly.express as px
import pandas as pd
from branca.element import IFrame

# =========================
# 1. SETUP DATA
# =========================

latest_quarter = "2025-Q3"
df_latest = df[df['quarter'] == latest_quarter].copy()
df['dt'] = pd.to_datetime(df['dt'], errors='coerce')

# =========================
# 2. LOAD GEOJSON
# =========================

geojson_url = "https://raw.githubusercontent.com/leakyMirror/map-of-europe/master/GeoJSON/europe.geojson"
geojson_data = requests.get(geojson_url).json()

# =========================
# 3. CREATE MAP
# =========================

m = folium.Map(
location=[54, 15],
    zoom_start=4,
    tiles='cartodbpositron',
    
    zoom_control=False,
    scrollWheelZoom=False,
    dragging=False,
   
    max_bounds=True,
    min_lat=33,
    max_lat=72,
    min_lon=-25,
    max_lon=45,
    min_zoom=4
)

m.fit_bounds([[34.0, -25.0], [72.0, 45.0]])

# =========================
# 4. CHOROPLETH (BASE)
# =========================

folium.Choropleth(
    geo_data=geojson_data,
    data=df_latest,
    columns=["country", "price_index"],
    key_on="feature.properties.NAME",
    fill_color="RdYlGn_r",
    fill_opacity=0.7,
    line_opacity=0.2,
    nan_fill_opacity=0.0
).add_to(m)

# =========================
# 5. ADD INTERACTIVE POPUPS (FIXED)
# =========================

for feature in geojson_data['features']:
    country_name = feature['properties']['NAME']
    history = df[df['country'] == country_name].sort_values('dt')

    if history.empty:
        continue

    # Create plotly chart
    fig = px.line(
        history,
        x='dt',
        y='price_index',
        title=country_name,
        markers=True,
        template='plotly_white'
    )

    fig.update_layout(
        width=350,
        height=250,
        margin=dict(l=10, r=10, t=30, b=10)
    )

    html = fig.to_html(include_plotlyjs='cdn', full_html=False)

    iframe = IFrame(html=html, width=370, height=270)
    popup = folium.Popup(iframe, max_width=400)

    # Attach popup to that specific country
    folium.GeoJson(
        feature,
        style_function=lambda x: {
            'fillColor': 'transparent',
            'color': 'transparent',
            'weight': 0
        },
        tooltip=country_name,
        popup=popup
    ).add_to(m)

# =========================
# 6. DISPLAY
# =========================

m
m.save('european_housing_prices_folium.html')